<div>
    <table>
        <tr>
            <td>
                <center>
                    <h1>Premise Introduction</h1>
                     <a href="https://www.psi.ch/en/ta/people/romain-sacchi">Romain Sacchi</a> (PSI)
                    <br><br>
                    Duration: 1 hour 15 minutes.
                </center>
            </td>
        </tr>
    </div>

<div class="alert alert-info">
Note: we will be using <a href="https://docs.brightway.dev/en/latest/content/installation/index.html">Brightway 2</a>, not <a href="https://docs.brightway.dev/en/legacy/">Brightway 2.5</a>, because we would like to visualize our results in <a href="https://github.com/LCA-ActivityBrowser/activity-browser">Activity Browser</a>, which is for now only compatible with Brightway 2. But if you do not require ``activity-browser``, you can run this notebook in an environment containing ``brightway25``.
</div>

In [1]:
from premise import __version__
__version__

/Users/qtu-ubc/anaconda3/envs/brightway25/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


14:26:17-0700 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


(2, 3, 9)

In [42]:
from premise import *
import bw2data as bd
import bw2io as bi
import bw2calc as bc

/Users/qtu-ubc/anaconda3/envs/brightway25/lib/python3.11/site-packages/bw2calc/__init__.py:53: UserWarning: 
It seems like you have an ARM architecture, but haven't installed scikit-umfpack:

    https://pypi.org/project/scikit-umfpack/

Installing it could give you much faster calculations.

  warnings.warn(UMFPACK_WARNING)


Link to Premise scenario dashboard: https://premisedash-6f5a0259c487.herokuapp.com/

Premise documentation: https://premise.readthedocs.io/en/latest/

Examples notebook: https://github.com/polca/premise/blob/master/examples/examples.ipynb

In [3]:
NewDatabase?

Init signature:
NewDatabase(
    scenarios: List[dict],
    source_version: str = '3.12',
    source_type: str = 'brightway',
    key: Union[bytes, str] = None,
    source_db: str = None,
    source_file_path: str = None,
    additional_inventories: List[dict] = None,
    system_model: str = 'cutoff',
    system_args: dict = None,
    use_cached_inventories: bool = True,
    use_cached_database: bool = True,
    external_scenarios: list = None,
    quiet=False,
    keep_imports_uncertainty=True,
    keep_source_db_uncertainty=False,
    gains_scenario='CLE',
    use_absolute_efficiency=False,
    biosphere_name: str = 'biosphere3',
    generate_reports: bool = True,
) -> None
Docstring:     
Class that represents a new wurst inventory database, modified according to IAM data.

:ivar source_type: the source of the ecoinvent database. Can be `brigthway` or `ecospold`.
:vartype source_type: str
:vartype source_db: str
:ivar system_model: Can be `cutoff` (default) or `consequential`.
:vart

In [3]:
bd.projects.set_current("BEST502")

In [4]:
bd.databases

Databases dictionary with 6 object(s):
	biosphere3
	carbon fiber
	ecoinvent-3.10-biosphere
	ecoinvent-3.10.1-cutoff
	ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20
	super_db_20-03-2026

In [5]:
# 3) Set the biosphere database name BEFORE creating ndb
from bw2data import config, databases

print(list(databases))
config.p["biosphere_database"] = "ecoinvent-3.10-biosphere"
print("biosphere setting:", config.p["biosphere_database"])

['ecoinvent-3.10-biosphere', 'ecoinvent-3.10.1-cutoff', 'carbon fiber', 'biosphere3', 'ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20', 'super_db_20-03-2026']
biosphere setting: ecoinvent-3.10-biosphere


In [6]:
# Run a quick sanity check first
from bw2data import Database, databases

print("source db exists:", "ecoinvent-3.10.1-cutoff" in databases)
print("biosphere db exists:", "ecoinvent-3.10-biosphere" in databases)
print("biosphere size:", len(Database("ecoinvent-3.10-biosphere")))

source db exists: True
biosphere db exists: True
biosphere size: 4362


In [11]:
ndb = NewDatabase(
    scenarios=[
        {"model":"image", "pathway":"SSP2-M", "year":2050},
        {"model":"remind", "pathway":"SSP2-NDC", "year":2050},
    ],
    source_db="ecoinvent-3.10.1-cutoff", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.10", # <-- version of ecoinvent. Can be "3.8", "3.9" or "3.10". Must be a string.
    key='tUePmX_S5B8ieZkkM7WUU2CnO8SmShwmAeWK9x2rTFo=', # <-- decryption key - used to assess default scenarios included in `premise`
    #The key is copied from the official Premise tutorial, available at: https://github.com/polca/premise/blob/master/examples/examples.ipynb
    biosphere_name="ecoinvent-3.10-biosphere",
)

ndb.biosphere_name = "ecoinvent-3.10-biosphere"
print("ndb biosphere:", ndb.biosphere_name)

premise v.(2, 3, 9)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from premise_gwp import add_premise_gwp                          |
| add_premise_gwp()                                                |
+------------------------------------------------------------------+
+-------------

In [16]:
# Peek at the first dataset and its first few exchanges to see how inputs are stored
first_ds = ndb.database[0]
print("Dataset keys:", first_ds.keys())
print("Dataset name:", first_ds.get("name"))
print("Number of exchanges:", len(first_ds.get("exchanges", [])))

for exc in first_ds.get("exchanges", [])[:5]:
    print(exc)
    print("---")

Dataset keys: dict_keys(['comment', 'location', 'name', 'reference product', 'unit', 'exchanges'])
Dataset name: treatment of municipal solid waste, sanitary landfill
Number of exchanges: 1
{'uncertainty type': 0, 'loc': 1.0, 'amount': 1.0, 'type': 'production', 'production volume': 22428450.0, 'product': 'heat, district or industrial, other than natural gas', 'name': 'treatment of municipal solid waste, sanitary landfill', 'unit': 'megajoule', 'location': 'CH'}
---


In [14]:
# select sector to integrate, or all sectors. for example, updating the electricity sector: ndb.update_electricity()
ndb.update("electricity")

Processing scenarios for sector 'electricity': 100%|█| 2/2 [00:17<00:0

Done!



In [17]:
ndb.write_db_to_brightway()

Write new database(s) to Brightway.
Running all checks...
Minor anomalies found: check the change report.
Database ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20 already exists: it will be overwritten.
14:49:50-0700 [info     ] Vacuuming database            
14:50:03-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|███████████████████████████████████████████████████████████████████████████| 28176/28176 [00:20<00:00, 1405.63it/s]


14:50:23-0700 [info     ] Vacuuming database            
Created database: ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20
Running all checks...
Minor anomalies found: check the change report.
14:51:03-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|███████████████████████████████████████████████████████████████████████████| 27200/27200 [00:19<00:00, 1375.34it/s]

14:51:23-0700 [info     ] Vacuuming database            


Created database: ei_cutoff_3.10_remind_SSP2-NDC_2050 2026-03-20
Generate scenario report.
Report saved under /Users/qtu-ubc/Downloads/export/scenario_report.
Generate change report.
Report saved under /Users/qtu-ubc/Downloads/export/change reports/.


In [19]:
ndb.write_superstructure_db_to_brightway()

Running all checks...
Minor anomalies found: check the change report.
Running all checks...
Minor anomalies found: check the change report.
Building superstructure database...
Dropped 2219 duplicate(s).
Scenario difference file exported to /Users/qtu-ubc/Downloads/export/scenario diff files!
Running all checks...
Minor anomalies found: check the change report.
Database super_db_20-03-2026 already exists: it will be overwritten.
14:55:03-0700 [info     ] Vacuuming database            
14:55:19-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|███████████████████████████████████████████████████████████████████████████| 26397/26397 [00:19<00:00, 1371.89it/s]

14:55:39-0700 [info     ] Vacuuming database            


Created database: super_db_20-03-2026
Generate scenario report.
Report saved under /Users/qtu-ubc/Downloads/export/scenario_report.
Generate change report.
Report saved under /Users/qtu-ubc/Downloads/export/change reports/.


## Check for changes

In [24]:
source = Database("ecoinvent-3.10.1-cutoff")
scenario = Database("ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20")

print(len(source), len(scenario))

23523 28176


## Test pLCA calculation

In [28]:
imp = bi.ExcelImporter("lci-carbon-fiber2.xlsx")

Extracted 1 worksheets in 0.03 seconds


In [33]:
bd.projects.set_current("BEST502")

In [47]:
bd.databases

Databases dictionary with 7 object(s):
	biosphere3
	carbon fiber
	ecoinvent-3.10-biosphere
	ecoinvent-3.10.1-cutoff
	ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20
	ei_cutoff_3.10_remind_SSP2-NDC_2050 2026-03-20
	super_db_20-03-2026

In [46]:
if "carbon fiber_plca" in bd.databases:
    del bd.databases["carbon fiber_plca"]

In [48]:
imp.match_database("ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20", fields=('name', 'reference product', 'location'))
imp.match_database("ecoinvent-3.10-biosphere", fields=('name', 'unit', 'categories'))
imp.statistics()

Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Graph statistics for `carbon fiber_plca` importer:
1 graph nodes:
	process: 1
7 graph edges:
	technosphere: 6
	production: 1
7 edges to the following databases:
	ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20: 7
0 unique unlinked edges (0 total):




(1, 7, 0, 0)

In [49]:
# Fix production exchanges so they point to the activity itself, not the background DB
for ds in imp.data:
    ds_code = ds["code"]
    ds_db = ds["database"]

    for exc in ds.get("exchanges", []):
        if exc.get("type") == "production":
            exc["input"] = (ds_db, ds_code)
            exc["database"] = ds_db

In [50]:
import uuid

for ds in imp.data:
    ds.setdefault("code", str(uuid.uuid4()))

imp.write_database()

18:19:06-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2624.72it/s]

18:19:06-0700 [info     ] Vacuuming database            


Created database: carbon fiber_plca


In [51]:
bd.databases

Databases dictionary with 8 object(s):
	biosphere3
	carbon fiber
	carbon fiber_plca
	ecoinvent-3.10-biosphere
	ecoinvent-3.10.1-cutoff
	ei_cutoff_3.10_image_SSP2-M_2050 2026-03-20
	ei_cutoff_3.10_remind_SSP2-NDC_2050 2026-03-20
	super_db_20-03-2026

In [52]:
# Verify that the rewritten foreground activity has a correct self-referential production exchange
db = bd.Database("carbon fiber_plca")
act = next(iter(db))
prods = list(act.production())
print(prods[0].as_dict())

{'name': 'polyacrylonitrile production (PAN) by polymerisation', 'amount': 1, 'database': 'carbon fiber_plca', 'location': 'RER', 'unit': 'kilogram', 'type': 'production', 'reference product': 'polyacrylonitrile', 'input': ('carbon fiber_plca', 'fa6201e3-5d8e-48ca-979d-8a40c3e838e8'), 'output': ('carbon fiber_plca', 'fa6201e3-5d8e-48ca-979d-8a40c3e838e8')}


In [53]:
ef_gwp_key = [
    m for m in bd.methods if "climate change" == m[1] and "TRACI v2.1" == m[0]
].pop()
print(ef_gwp_key)

# We can look at the method details
ef_gwp_method = bd.Method(ef_gwp_key)
print(ef_gwp_method)

('TRACI v2.1', 'climate change', 'global warming potential (GWP100)')
Brightway2 Method: TRACI v2.1: climate change: global warming potential (GWP100)


In [54]:
wb = bd.Database("carbon fiber_plca")
our_activity = [act for act in wb][0]
print(our_activity)

'polyacrylonitrile production (PAN) by polymerisation (plca)' (kilogram, RER, None)


In [55]:
act = next(iter(db))

print("Activity:", act["name"])
print("Reference product:", act.get("reference product"))
print("Unit:", act.get("unit"))

prods = list(act.production())
print("Number of production exchanges:", len(prods))

for exc in prods:
    print(exc.as_dict())

Activity: polyacrylonitrile production (PAN) by polymerisation (plca)
Reference product: polyacrylonitrile
Unit: kilogram
Number of production exchanges: 1
{'name': 'polyacrylonitrile production (PAN) by polymerisation', 'amount': 1, 'database': 'carbon fiber_plca', 'location': 'RER', 'unit': 'kilogram', 'type': 'production', 'reference product': 'polyacrylonitrile', 'input': ('carbon fiber_plca', 'fa6201e3-5d8e-48ca-979d-8a40c3e838e8'), 'output': ('carbon fiber_plca', 'fa6201e3-5d8e-48ca-979d-8a40c3e838e8')}


In [56]:
my_functional_unit, data_objs, _ = bd.prepare_lca_inputs(
    {our_activity: 1},
    method=ef_gwp_key,
)
my_lca = bc.LCA(demand=my_functional_unit, data_objs=data_objs)
my_lca.lci()
my_lca.lcia()
my_lca.score

10.186326566926807

In [63]:
""" compare with benchmark (ecoinvent 3.10.1 as-is) """

imp_bench = bi.ExcelImporter("lci-carbon-fiber2.xlsx")

# Rename foreground DB for benchmark version
for ds in imp_bench.data:
    old_db = ds["database"]
    ds["database"] = "carbon fiber_plca_benchmark"
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "production":
            exc["database"] = "carbon fiber_plca_benchmark"

# Benchmark = original ecoinvent
imp_bench.match_database(
    "ecoinvent-3.10.1-cutoff",
    fields=("name", "reference product", "location"),
)
imp_bench.match_database("ecoinvent-3.10-biosphere", fields=('name', 'unit', 'categories'))

for ds in imp_bench.data:
    ds.setdefault("code", str(uuid.uuid4()))
    
# Fix production exchanges so they point to self
def fix_production_self_links(imp):
    for ds in imp.data:
        ds_code = ds["code"]
        ds_db = ds["database"]
        for exc in ds.get("exchanges", []):
            if exc.get("type") == "production":
                exc["input"] = (ds_db, ds_code)
                exc["output"] = (ds_db, ds_code)
                exc["database"] = ds_db

fix_production_self_links(imp_bench)

# write benchmark database
imp_bench.write_database(db_name="carbon fiber_plca_benchmark")

# calculate impacts for benchmark
bench_db = bd.Database("carbon fiber_plca_benchmark")
bench_act = next(iter(bench_db))

fu_bench, data_objs_bench, _ = bd.prepare_lca_inputs(
    {bench_act: 1},
    method=ef_gwp_key,
)

lca_bench = bc.LCA(demand=fu_bench, data_objs=data_objs_bench)
lca_bench.lci()
lca_bench.lcia()

print("Benchmark:", lca_bench.score)

Extracted 1 worksheets in 0.02 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
19:41:16-0700 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|███████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5102.56it/s]

19:41:16-0700 [info     ] Vacuuming database            


Created database: carbon fiber_plca_benchmark
Benchmark: 11.65369017399957


# You can stop here

## Incremental Databases

The class ``IncrementalDatabase`` also implementing sectoral changes in an incremental, to easily distinguish the contribution of each sector on the results.

In [21]:
ndb = IncrementalDatabase(
    scenarios=[
        {"model":"image", "pathway":"SSP2-RCP26", "year":2040},
    ],
    source_db="ecoinvent-3.10.1-cutoff", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.10", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.8". Must be a string.
    #key="xxxxxxxxxxxxxxxxxx",
)

Reading unencrypted IAM output files.
Cannot find the IAM scenario file at /Users/qtu-ubc/anaconda3/envs/brightway25/lib/python3.11/site-packages/premise/data/iam_output_files/image_SSP2-RCP26. Will check online.
premise v.(2, 3, 9)
+------------------------------------------------------------------+
| Warning                                                          |
+------------------------------------------------------------------+
| Because some of the scenarios can yield LCI databases            |
| containing net negative emission technologies (NET),             |
| it is advised to account for biogenic CO2 flows when calculating |
| Global Warming potential indicators.                             |
| `premise_gwp` provides characterization factors for such flows.  |
| It also provides factors for hydrogen emissions to air.          |
|                                                                  |
| Within your Brightway project:                                   |
| from p

FileNotFoundError: Either 1) the file image_SSP2-RCP26 cannot found with any supported extension in /Users/qtu-ubc/anaconda3/envs/brightway25/lib/python3.11/site-packages/premise/data/iam_output_filesor 2) no decryption key provided to download the file from Zenodo. Please provide a decryption key or place the file in the specified directory.

In [ ]:
sectors = {
    "electricity": "electricity",
    "steel": "steel",
    "others": [
        "cement",
        "cars",
        "fuels"
    ]
}

In [ ]:
ndb.update(sectors=sectors)

In [ ]:
ndb.write_increment_db_to_brightway(name="test increment", file_format="csv")

## Reports
### Scenario report
You can generate a spreadsheet report showing the main variables of the scenario you have selected to create your databases. The report is saved in your working directory. Note that this report is generated automatically when exporting a database.

In [22]:
ndb.generate_scenario_report()

Generate scenario report.
Report saved under /Users/qtu-ubc/Downloads/export/scenario_report.


## Changes report
You can generate a spreadsheet report of the changes made to the original database. It gives an overview on:
* the datasets created
* the datasets modified
* some performance indicators
* scaling factors used to scale certain exchanges

A "Validation" tab also shows any datasets that contain values or efficiencies that may seem incorrect.

The report is saved in your working directory. It is generated automatically when you export a database.

In [23]:
ndb.generate_change_report()

Generate change report.
Report saved under /Users/qtu-ubc/Downloads/export/change reports/.


## Custom scenarios

You can also create your own scenarios by providing a data package to premise. This data package should contain the following files:
* ``datapackage.json``: a json file containing the metadata of the data package
* ``scenario_data.csv``: a csv file containing the scenario data
* ``config.yaml``: a yaml file containing the configuration of the scenario (which markets to create, etc.)
* ``lci-inventories.csv``: a csv file containing additional inventories to import

In [ ]:
import brightway2 as bw
from premise import NewDatabase
from datapackage import Package


fp = r"https://raw.githubusercontent.com/premise-community-scenarios/ammonia-prospective-scenarios/main/datapackage.json"
ammonia = Package(fp)

bw.projects.set_current("your_bw_project")

ndb = NewDatabase(
    scenarios = [
        {"model":"image", "pathway":"SSP2-Base", "year":2050, "external scenarios": [{"scenario": "Business As Usual - image", "data": ammonia}]},
        {"model":"image", "pathway":"SSP2-RCP26", "year":2030, "external scenarios": [{"scenario": "Sustainable development - image", "data": ammonia}]},
    ],        
    source_db="ecoinvent 3.10 cutoff",
    source_version="3.10",
    key='xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx',
)
    
ndb.update("external") # or ndb.update() if you want to update the database with the IAM data plus the external scenario
